# 🌍 GeoPulse - RAG + Gemini 분석 파이프라인
### AI 기반 지정학 분쟁 분석 플랫폼
---
**실행 전 확인:**
- ✅ `GeoPulse_DL.ipynb` 실행 완료 (models/ 폴더 생성됨)
- ✅ `.env` 파일에 `GEMINI_API_KEY` 설정됨
- ✅ `./data/GeoPulse_Final_Dataset_KOREAN.csv` 존재

**실행 순서:** 셀을 위에서 아래로 순서대로 실행하세요

In [1]:
# ============================================================
# 셀 1: 라이브러리 & 설정
# 초심자 핵심: RAG에 필요한 도구
#   SentenceTransformer → 문장을 숫자 벡터로 변환
#   faiss               → 벡터 간 거리 초고속 검색
#   google.generativeai → Gemini API 호출
# ============================================================
import os
import pandas as pd
import numpy as np
import pickle
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss
from dotenv import load_dotenv

# DL 모델 로드용
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

load_dotenv()

# API 키 설정
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# 전역 설정
EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'  # 한국어 지원
TOP_K       = 5
MAX_LEN     = 30
DATA_PATH   = './data/GeoPulse_Final_Dataset_KOREAN.csv'
INDEX_PATH  = 'rag/faiss_index.bin'
META_PATH   = 'rag/metadata.pkl'

os.makedirs('rag', exist_ok=True)
print('✅ 환경 설정 완료')
print(f'   API 키 확인: {"✅ 설정됨" if GEMINI_API_KEY != "여기에_API키_직접_입력" else "❌ 미설정 - .env 파일 확인 필요"}')

d:\LLM-DL\venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
d:\LLM-DL\venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
d:\LLM-DL\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\human-27\AppData\Local\Temp\ipykernel_12552\2785426276.py:12: FutureWarning: 

All support for the `google.generativeai` packa


✅ 환경 설정 완료
   API 키 확인: ✅ 설정됨


## 📦 벡터 저장소 클래스 (FAISS)

In [11]:
# ============================================================
# 셀 2: ConflictVectorStore 클래스
# 초심자 핵심:
#   텍스트를 숫자(벡터)로 바꿔서 '지도'처럼 저장
#   나중에 비슷한 분쟁을 이 지도의 거리를 보고 찾는다
# ============================================================
class ConflictVectorStore:
    """
    RAG 파이프라인의 검색 엔진
    1. build()  → CSV를 벡터로 변환 후 FAISS 저장
    2. load()   → 저장된 인덱스 불러오기
    3. search() → 입력 쿼리와 유사한 분쟁 사례 검색
    """
    def __init__(self):
        print('📥 임베딩 모델 로딩 중... (최초 실행 시 다운로드됨)')
        self.embedder = SentenceTransformer(EMBED_MODEL)
        self.index    = None
        self.metadata = []
        print('✅ 임베딩 모델 로드 완료')

    def _make_doc(self, row) -> str:
        """행 → 검색용 문서 텍스트 변환 (검색 품질 결정)"""
        return (
            f"발생지: {row.get('발생지', '')} | "
            f"교전A: {row.get('정부군(교전A)', '')} | "
            f"교전B: {row.get('반군/적대측(교전B)', '')} | "
            f"원인: {row.get('분쟁원인', '')} | "
            f"강도: {row.get('전쟁강도', '')} | "
            f"연도: {row.get('연도', '')}"
        )

    def build(self, csv_path: str):
        """CSV → 임베딩 → FAISS 인덱스 구축 & 저장"""
        df   = pd.read_csv(csv_path)
        docs = [self._make_doc(row) for _, row in df.iterrows()]

        print(f'🚀 {len(docs)}개 문서 임베딩 중... (1~2분 소요)')
        embeddings = self.embedder.encode(
            docs,
            convert_to_numpy=True,
            normalize_embeddings=True,  # 코사인 유사도용 정규화
            show_progress_bar=True
        )

        # FAISS 인덱스 생성 (Inner Product = 코사인 유사도)
        dim        = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings.astype(np.float32))

        # 메타데이터 저장 (검색 결과에서 원본 데이터 참조용)
        self.metadata = df.to_dict(orient='records')
        faiss.write_index(self.index, INDEX_PATH)
        with open(META_PATH, 'wb') as f:
            pickle.dump(self.metadata, f)

        print(f'\n✅ 벡터 저장소 구축 완료!')
        print(f'   총 {self.index.ntotal}개 벡터 저장됨')
        print(f'   인덱스: {INDEX_PATH}')
        print(f'   메타데이터: {META_PATH}')

    def load(self):
        """저장된 인덱스 로드 (build 이후 재실행 시 사용)"""
        if not os.path.exists(INDEX_PATH):
            raise FileNotFoundError('인덱스 없음! build()를 먼저 실행하세요.')
        self.index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, 'rb') as f:
            self.metadata = pickle.load(f)
        print(f'✅ 인덱스 로드 완료: {self.index.ntotal}개 벡터')

    def search(self, query: str, top_k: int = TOP_K) -> list:
        """쿼리와 가장 유사한 분쟁 사례 검색"""
        q_vec = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.index.search(q_vec, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.metadata):
                item = dict(self.metadata[idx])
                item['similarity'] = float(score)
                results.append(item)
        return results

print('✅ ConflictVectorStore 정의 완료')

✅ ConflictVectorStore 정의 완료


## 🤖 Gemini 리포트 생성기

In [ ]:
# ============================================================
# 셀 3: GeoReportGenerator 클래스
# 초심자 핵심:
#   Gemini한테 검색된 데이터 + DL 결과를 주고
#   '지정학 전문가처럼 리포트 써라'고 시키는 부분
#   프롬프트 엔지니어링이 리포트 퀄리티를 결정한다!
# ============================================================
class GeoReportGenerator:
    """   
    리포트 섹션:
    1. 공식 명분 분석
    2. AI 추론: 숨겨진 이유
    3. 주변국 & 동맹국 관계망
    4. 역사적 패턴 분석
    5. 전쟁 가능성 & 시나리오
    6. 피해 & 이득 분석
    7. 종합 전망
    """
    def __init__(self):
        genai.configure(api_key=GEMINI_API_KEY)
        # gemini-2.0-flash: 무료 티어 지원, 빠른 응답
        self.model = genai.GenerativeModel('gemini-2.5-flash')
        print('✅ Gemini 모델 초기화 완료 (gemini-2.5-flash)')

    def _build_prompt(self, data: dict) -> str:
        """리포트 생성용 프롬프트 구성"""
        # 유사 사례 텍스트 변환
        cases_text = '\n'.join([
            f"  {i+1}. {c.get('발생지','')} ({c.get('연도','')}) | "
            f"원인: {c.get('분쟁원인','')} | "
            f"강도: {c.get('전쟁강도','')} | "
            f"사망자: {c.get('사망자_추정치','')}"
            for i, c in enumerate(data.get('similar_cases', [])[:5])
        ])

        return f"""당신은 국제정치 및 지정학 전문 분석가입니다.
아래 정보를 바탕으로 GeoPulse 종합 분쟁 분석 리포트를 작성하세요.

=== 분쟁 기본 정보 ===
분쟁명: {data.get('name', '')}
지역: {data.get('region', '')}
공식 원인: {data.get('official_cause', '')}
전쟁 강도: {data.get('intensity', '')}
추정 사망자: {data.get('deaths', '')}명
추가 맥락: {data.get('context', '')}

=== AI 딥러닝 분석 결과 ===
분쟁원인 분류: {data.get('dl_cause', '')} (신뢰도 {data.get('dl_cause_conf', 0):.1f}%)
전쟁강도 분류: {data.get('dl_risk', '')} (신뢰도 {data.get('dl_risk_conf', 0):.1f}%)

=== 역사적 유사 사례 (RAG 검색) ===
{cases_text}

=== 작성 지침 ===
다음 7개 섹션으로 리포트를 작성하세요.

1. 📋 공식 명분 분석
   - 표면적으로 제시된 전쟁 원인

2. 🔍 AI 추론: 숨겨진 이유
   - 자원, 지정학, 패권, 경제 관점에서 실제 이유 분석
   - 수혜 예상 세력 명시

3. 🌐 주변국 & 동맹국 관계망
   - 지원국: 이유 + 예상 이득
   - 반대국: 이유 + 예상 손해
   - 중립국: 이유 + 전략적 입장

4. 📜 역사적 패턴 분석
   - 유사 사례와 비교
   - 역사 패턴 설명

5. ⚡ 전쟁 가능성 & 시나리오
   - 전면전 가능성: XX%
   - 시나리오 A (교전A 승리) / B (교전B 승리) / C (교착)

6. 💰 피해 & 이득 분석
   - 인적/경제적 피해, 예상 쟁취물, 이후 활용 예측

7. 🔮 종합 전망
   - 핵심 결론 3줄 요약

마지막에 반드시 추가:
⚠️ 본 리포트는 AI 추론 기반 분석으로, 공식 입장과 다를 수 있습니다.
"""

    def generate(self, data: dict) -> str:
        """리포트 생성"""
        prompt   = self._build_prompt(data)
        print('🤖 Gemini 리포트 생성 중...')
        response = self.model.generate_content(prompt)
        return response.text

print('✅ GeoReportGenerator 정의 완료')

✅ GeoReportGenerator 정의 완료


## 🔗 통합 파이프라인 클래스 (DL + RAG + Gemini)

In [ ]:
# ============================================================
# 셀 4: GeoPulseAnalyzer - 통합 파이프라인
# 초심자 핵심:
#   DL 예측 → RAG 검색 → Gemini 리포트 생성
#   이 세 단계를 하나로 묶어주는 핵심 클래스
# ============================================================
class GeoPulseAnalyzer:
    """
    통합 분석 파이프라인
    
    흐름:
    입력(분쟁명, 지역, 원인 등)
      → DL: 원인/강도 분류
      → RAG: 유사 역사 사례 검색
      → Gemini: 종합 리포트 생성
    """
    def __init__(self):
        self.vector_store  = None
        self.reporter      = None
        self.cause_model   = None
        self.risk_model    = None
        self.tokenizer     = None
        self.cause_rev     = {0: '영토', 1: '정부/권력', 2: '복합'}
        self.risk_rev      = {0: '소규모', 1: '전면전'}

    def setup_rag(self, build_index: bool = False):
        """RAG 벡터 저장소 초기화"""
        self.vector_store = ConflictVectorStore()
        if build_index:
            self.vector_store.build(DATA_PATH)
        else:
            self.vector_store.load()

    def setup_gemini(self):
        """Gemini 리포트 생성기 초기화"""
        self.reporter = GeoReportGenerator()

    def setup_dl(self):
        """DL 모델 & 토크나이저 로드"""
        try:
            self.cause_model = load_model('models/cause_classifier.keras')
            self.risk_model  = load_model('models/risk_classifier.keras')
            with open('models/tokenizer.pkl', 'rb') as f:
                self.tokenizer = pickle.load(f)
            print('✅ DL 모델 로드 완료')
        except Exception as e:
            print(f'⚠️ DL 모델 미로드: {e}')
            print('   GeoPulse_DL.ipynb 먼저 실행하세요')

    def _predict_dl(self, query_text: str) -> tuple:
        """DL로 원인/강도 예측"""
        if self.cause_model is None:
            return ({'label': '미분류', 'confidence': 0.0},
                    {'label': '미분류', 'confidence': 0.0})

        seq    = self.tokenizer.texts_to_sequences([query_text])
        padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')

        c_probs = self.cause_model.predict(padded, verbose=0)[0]
        r_probs = self.risk_model.predict(padded, verbose=0)[0]

        c_idx = int(np.argmax(c_probs))
        r_idx = int(np.argmax(r_probs))

        cause_result = {'label': self.cause_rev[c_idx],
                        'confidence': float(c_probs[c_idx])}
        risk_result  = {'label': self.risk_rev[r_idx],
                        'confidence': float(r_probs[r_idx])}
        return cause_result, risk_result

    def analyze(self, conflict_name: str, region: str,
                official_cause: str, intensity: str,
                deaths: str = '0', context: str = '') -> dict:
        """분쟁 종합 분석 메인 함수"""

        # 1) DL 예측
        query_text   = f'{conflict_name} {region} {official_cause} {context}'
        cause_result, risk_result = self._predict_dl(query_text)

        # 2) RAG 유사 사례 검색
        similar_cases = self.vector_store.search(query_text, top_k=TOP_K)

        # 3) Gemini 리포트 생성
        data = {
            'name':          conflict_name,
            'region':        region,
            'official_cause': official_cause,
            'intensity':     intensity,
            'deaths':        deaths,
            'context':       context,
            'dl_cause':      cause_result['label'],
            'dl_cause_conf': cause_result['confidence'] * 100,
            'dl_risk':       risk_result['label'],
            'dl_risk_conf':  risk_result['confidence'] * 100,
            'similar_cases': similar_cases,
        }
        report = self.reporter.generate(data)

        return {
            'dl_cause':     cause_result,
            'dl_risk':      risk_result,
            'similar_cases': similar_cases,
            'report':       report,
        }

print('✅ GeoPulseAnalyzer 정의 완료')

✅ GeoPulseAnalyzer 정의 완료


## 🚀 초기화 실행

In [14]:
# ============================================================
# 셀 5: 벡터 저장소 구축
# 초심자 핵심:
#   처음 한 번만 build_index=True 로 실행!
#   rag/ 폴더에 인덱스 저장되면 다음부터 False로 실행
# ============================================================
analyzer = GeoPulseAnalyzer()

# ★ 처음 실행: True / 이미 인덱스 있으면: False
analyzer.setup_rag(build_index=True)

📥 임베딩 모델 로딩 중... (최초 실행 시 다운로드됨)
✅ 임베딩 모델 로드 완료
🚀 1586개 문서 임베딩 중... (1~2분 소요)


Batches: 100%|██████████| 50/50 [00:28<00:00,  1.77it/s]


✅ 벡터 저장소 구축 완료!
   총 1586개 벡터 저장됨
   인덱스: rag/faiss_index.bin
   메타데이터: rag/metadata.pkl


In [15]:
# ============================================================
# 셀 6: Gemini & DL 모델 초기화
# ============================================================
analyzer.setup_gemini()
analyzer.setup_dl()
print('\n✅ 전체 초기화 완료! 분석 준비됨')

✅ Gemini 모델 초기화 완료 (gemini-2.5-flash)
✅ DL 모델 로드 완료

✅ 전체 초기화 완료! 분석 준비됨


## 🔍 RAG 검색 테스트

In [16]:
# ============================================================
# 셀 7: RAG 검색 단독 테스트
# 초심자 핵심: 검색이 잘 되는지 먼저 확인해라
# ============================================================
query = '러시아 우크라이나 돈바스 영토 분쟁'
similar = analyzer.vector_store.search(query, top_k=5)

print(f'🔍 검색어: {query}')
print('\n📚 유사 사례 TOP 5:')
print('=' * 60)
for i, c in enumerate(similar, 1):
    print(f"{i}. {c.get('발생지','')} ({c.get('연도','')})")
    print(f"   원인: {c.get('분쟁원인','')} | 강도: {c.get('전쟁강도','')}")
    print(f"   사망자: {c.get('사망자_추정치','')} | 유사도: {c.get('similarity',0):.3f}")
    print()

🔍 검색어: 러시아 우크라이나 돈바스 영토 분쟁

📚 유사 사례 TOP 5:
1. 러시아, 우크라이나 (2024)
   원인: 복합 | 강도: 전면전
   사망자: 75686 | 유사도: 0.804

2. 러시아, 우크라이나 (2023)
   원인: 복합 | 강도: 전면전
   사망자: 75180 | 유사도: 0.771

3. 러시아, 우크라이나 (2022)
   원인: 복합 | 강도: 전면전
   사망자: 91502 | 유사도: 0.739

4. 우크라이나 (2014)
   원인: 영토 | 강도: 전면전
   사망자: 1072 | 유사도: 0.696

5. 러시아 (2015)
   원인: 영토 | 강도: 소규모
   사망자: 55 | 유사도: 0.682



## 🤖 DL + RAG + Gemini 통합 분석

In [17]:
# ============================================================
# 셀 8: 통합 분석 실행 (러시아-우크라이나)
# 초심자 핵심: 여기가 진짜 핵심!
#   DL → RAG → Gemini 세 단계가 한 번에 실행됨
# ============================================================
result = analyzer.analyze(
    conflict_name  = '러시아-우크라이나 전쟁',
    region         = '동유럽',
    official_cause = '영토',
    intensity      = '전면전',
    deaths         = '500000',
    context        = 'NATO 확장, 흑해 패권, 에너지 자원, 돈바스'
)

# DL 결과 출력
print('=' * 60)
print('🤖 DL 분류 결과')
print('=' * 60)
print(f"분쟁원인: {result['dl_cause']['label']} "
      f"({result['dl_cause']['confidence']*100:.1f}%)")
print(f"전쟁강도: {result['dl_risk']['label']} "
      f"({result['dl_risk']['confidence']*100:.1f}%)")

🤖 Gemini 리포트 생성 중...
🤖 DL 분류 결과
분쟁원인: 영토 (65.2%)
전쟁강도: 소규모 (64.9%)


In [18]:
# ============================================================
# 셀 9: 리포트 출력
# ============================================================
print('=' * 60)
print('📝 GeoPulse 종합 분석 리포트')
print('=' * 60)
print(result['report'])

📝 GeoPulse 종합 분석 리포트
## GeoPulse 종합 분쟁 분석 리포트: 러시아-우크라이나 전쟁

**분석 일자:** 2024년 5월 20일
**분쟁명:** 러시아-우크라이나 전쟁
**지역:** 동유럽
**전쟁 강도:** 전면전
**추정 사망자:** 500,000명

---

### 1. 📋 공식 명분 분석

러시아는 표면적으로 우크라이나의 영토(특히 돈바스 지역)에 대한 역사적 권리 주장과 러시아어 사용 인구 보호를 명분으로 내세우고 있습니다. 또한, 우크라이나의 NATO 가입 추진을 자국 안보에 대한 직접적인 위협으로 간주하며 '비무장화' 및 '탈나치화'를 전쟁의 공식적인 목표로 제시했습니다. 반면, 우크라이나는 자국의 주권과 영토 보전을 수호하기 위한 방어 전쟁임을 천명하고 있으며, 이는 국제법적 정당성을 확보하려는 노력의 일환입니다.

### 2. 🔍 AI 추론: 숨겨진 이유

AI 딥러닝 분석은 분쟁 원인을 '영토'로 분류했으나 신뢰도(65.2%)가 상대적으로 낮아, 표면적인 영토 문제 이면에 복합적인 전략적 이해관계가 숨겨져 있음을 시사합니다.

*   **지정학적 패권:** 러시아는 NATO의 동진 확장을 자국 안보에 대한 심각한 위협으로 인식하며, 우크라이나를 서방과의 완충지대로 유지하고 궁극적으로 자국의 영향권 내에 두려는 강력한 의도를 가지고 있습니다. 흑해 패권 장악은 러시아 해군력 투사 및 무역로 확보에 필수적이며, 이는 크림 반도 합병과 연장선상에 있습니다.
*   **에너지 자원 및 경제:** 우크라이나를 통한 유럽으로의 에너지 파이프라인 통제권, 우크라이나 내 잠재된 자원(천연가스, 광물) 및 비옥한 농경지(세계 곡물 시장 영향력) 확보 또한 중요한 동기입니다.
*   **수혜 예상 세력:** 러시아는 우크라이나를 성공적으로 통제함으로써 서방의 영향력을 약화시키고 지역적 패권을 강화할 수 있습니다. 장기적으로는 유럽 에너지 시장에 대한 영향력을 유지하고, 군수 산업체 및 관련 기술 개발국가들이 전쟁 특수를 누릴 수 있습니다

In [19]:
# ============================================================
# 셀 10: 다른 분쟁 테스트 (이스라엘-하마스)
# 초심자 핵심: 여기서 입력값 바꿔가면서 테스트해봐라
# ============================================================
result2 = analyzer.analyze(
    conflict_name  = '이스라엘-하마스 전쟁',
    region         = '중동',
    official_cause = '영토',
    intensity      = '전면전',
    deaths         = '40000',
    context        = '가자지구, 종교 갈등, 팔레스타인 독립, 미국 지원'
)

print('=' * 60)
print('📝 이스라엘-하마스 분석 리포트')
print('=' * 60)
print(f"DL 원인: {result2['dl_cause']['label']} "
      f"({result2['dl_cause']['confidence']*100:.1f}%)")
print(f"DL 강도: {result2['dl_risk']['label']} "
      f"({result2['dl_risk']['confidence']*100:.1f}%)")
print()
print(result2['report'])

🤖 Gemini 리포트 생성 중...
📝 이스라엘-하마스 분석 리포트
DL 원인: 영토 (57.6%)
DL 강도: 소규모 (82.4%)

## GeoPulse 종합 분쟁 분석 리포트: 이스라엘-하마스 전쟁

---

### 1. 📋 공식 명분 분석

이스라엘-하마스 전쟁의 공식적인 명분은 "영토"에 뿌리를 두고 있습니다. 팔레스타인인들은 가자지구를 포함한 점령지 내에서의 독립 국가 수립과 자결권을 주장하며, 이스라엘은 자국의 안보와 국경 방위를 명분으로 가자지구 내 하마스 무장 세력의 위협 제거를 내세우고 있습니다. 이는 단순한 국경선 문제를 넘어, 양측의 존재론적 위협 인식과 생존권을 둘러싼 근본적인 영토 주권 갈등으로 표면화되고 있습니다.

### 2. 🔍 AI 추론: 숨겨진 이유

AI 딥러닝 분석은 분쟁 원인을 '영토'로 분류했으나, 신뢰도가 57.6%로 아주 높지는 않다는 점에서 공식적인 명분 외에 더 복잡한 이해관계가 얽혀 있음을 시사합니다. 실제로는 다음과 같은 지정학적, 패권적, 경제적 요인들이 복합적으로 작용하고 있습니다.

*   **지정학적 패권:** 이스라엘은 중동 지역 내에서 자국의 안보 우위를 확고히 하고, 이란의 영향력 확대를 견제하려는 전략적 목표를 가지고 있습니다. 하마스 제거는 이스라엘의 안보를 강화하고, 팔레스타인 자치정부의 위상을 약화시켜 장기적으로 이스라엘에 유리한 지역 질서를 구축하려는 의도가 내포되어 있습니다.
*   **자원:** 가자지구 인근 해상에는 상당한 규모의 천연가스 매장지가 존재합니다. 분쟁 이후 가자지구의 미래와 통제권은 이러한 잠재적 에너지 자원의 개발 및 이권 문제와 연결될 수 있습니다.
*   **경제적 압박 및 재건 이권:** 가자지구의 재건에는 막대한 국제 자본이 필요하며, 이는 향후 특정 세력이나 국가에 경제적 이득을 제공할 수 있습니다. 또한 이스라엘은 하마스 축출을 통해 장기적으로 가자지구 경제에 대한 간접적 영향력을 확대하려는 의도를 가질 수 있습니다.
*   **국내 정치적 요인:** 양

## 💾 분석기 저장 (Gradio 앱 연동용)

In [20]:
# ============================================================
# 셀 11: analyzer 객체 저장
# 초심자 핵심:
#   이걸 저장해야 app.py(Gradio)에서 불러다 쓸 수 있음
#   매번 초기화하면 느리니까 한 번 만들어서 재사용
# ============================================================
print('✅ RAG 파이프라인 완료!')
print()
print('📁 생성된 파일:')
print('   rag/faiss_index.bin  ← FAISS 벡터 인덱스')
print('   rag/metadata.pkl     ← 분쟁 메타데이터')
print()
print('▶ 다음 단계: GeoPulse_App.ipynb 실행 (Gradio UI)')

✅ RAG 파이프라인 완료!

📁 생성된 파일:
   rag/faiss_index.bin  ← FAISS 벡터 인덱스
   rag/metadata.pkl     ← 분쟁 메타데이터

▶ 다음 단계: GeoPulse_App.ipynb 실행 (Gradio UI)
